## PakMart Traders – Prediction / Batch Scoring

Loads the registered MLflow model and scores recent sales data to predict `TotalIncludingTax`.

In [ ]:
import mlflow
from pyspark.ml import Pipeline
from synapse.ml.core.platform import *
from synapse.ml.lightgbm import LightGBMRegressor

mlexperiment_name = 'pakmart_predict_sale_amount'
mlalgorithm_name  = 'lightgbm'
mlmodel_name      = f'{mlexperiment_name}_{mlalgorithm_name}'
mlmodel_uri       = f'models:/{mlmodel_name}/1'   # update version as needed

loaded_model = mlflow.spark.load_model(mlmodel_uri, dfs_tmpdir='Files/mlflow/tmp/')
print(f'Model loaded: {mlmodel_uri}')

In [ ]:
SEED = 42

# Score 2023 incremental data (25 % sample)
prediction_input_df = spark.table('pakmart_sale_clean') \
    .filter('Year = 2023') \
    .sample(True, 0.25, seed=SEED)

print(f'Scoring {prediction_input_df.count():,} rows ...')

In [ ]:
# Generate predictions
prediction_output_df = loaded_model.transform(prediction_input_df)

# Drop intermediate pipeline columns
cols_to_remove = [
    'SalesTerritoryStrIdx', 'BuyingGroupStrIdx', 'CategoryStrIdx',
    'DayOfWeekNameStrIdx', 'SeasonStrIdx',
    'SalesTerritoryOHEnc', 'BuyingGroupOHEnc', 'CategoryOHEnc',
    'DayOfWeekNameOHEnc', 'SeasonOHEnc', 'features'
]

prediction_output_df = prediction_output_df \
    .withColumnRenamed('prediction', 'PredictedTotalIncludingTax') \
    .drop(*cols_to_remove)

In [ ]:
display(prediction_output_df)

In [ ]:
table_name = 'pakmart_sale_prediction'
prediction_output_df.write.mode('overwrite').format('delta').saveAsTable(table_name)
print(f'Predictions saved to: {table_name}')

### Evaluate prediction quality

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

ev_rmse = RegressionEvaluator(labelCol='TotalIncludingTax', predictionCol='PredictedTotalIncludingTax', metricName='rmse')
ev_r2   = RegressionEvaluator(labelCol='TotalIncludingTax', predictionCol='PredictedTotalIncludingTax', metricName='r2')

print(f'Inference RMSE: {ev_rmse.evaluate(prediction_output_df):,.2f} PKR')
print(f'Inference R²:   {ev_r2.evaluate(prediction_output_df):.4f}')